# Session 1: Retrieval-Augmented Generation (Instructor Notebook)

Welcome to Day 3! In this session, students build production-grade RAG systems. Moving beyond simple keyword matching, they learn embedding-based retrieval, vector stores, advanced chunking, query transformation, and evaluation — the techniques that make RAG reliable in the real world.

> **Instructor Note:** This notebook contains all 5 demos (identical to student) plus fully solved versions of all 4 tasks with approach explanations.

## Learning Objectives

By the end of this session, students will be able to:

1. **Create and compare** text embeddings using OpenAI's embedding models
2. **Build and query** a vector store using ChromaDB
3. **Apply advanced chunking** strategies for better retrieval quality
4. **Transform queries** using HyDE and multi-query techniques
5. **Build an end-to-end RAG pipeline** with source citations
6. **Evaluate RAG quality** with retrieval and generation metrics

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import chromadb
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.messages import HumanMessage, SystemMessage
import numpy as np
import json
import os

print("All imports successful!")

---
## Demos (Follow Along)

### Demo 1: Embedding Models — Creating and Comparing Text Embeddings

Embeddings convert text into numerical vectors in a high-dimensional space where **semantically similar texts are close together**. This is the foundation of all modern retrieval systems.

In [ ]:
# Demo 1 - Embedding Models

client = openai.OpenAI()

# Step 1: Create embeddings for sample texts
texts = [
    "Python is a popular programming language.",
    "Java is widely used in enterprise software.",
    "Machine learning models can classify images.",
    "Deep learning is a subset of machine learning.",
    "The weather today is sunny and warm."
]

response = client.embeddings.create(
    model="text-embedding-3-small",
    input=texts
)

embeddings = [item.embedding for item in response.data]
print(f"Number of embeddings: {len(embeddings)}")
print(f"Embedding dimensions: {len(embeddings[0])}")
print(f"First 5 values of embedding 0: {embeddings[0][:5]}")

# Step 2: Compute cosine similarity
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("\nSimilarity matrix:")
print(f"{'':>45}", end="")
for i in range(len(texts)):
    print(f"  [{i}]", end="")
print()
for i, t1 in enumerate(texts):
    print(f"[{i}] {t1[:40]:>42}", end="")
    for j, t2 in enumerate(texts):
        sim = cosine_similarity(embeddings[i], embeddings[j])
        print(f" {sim:.2f}", end="")
    print()

# Step 3: Semantic search
query = "What programming languages are popular?"
query_emb = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding

print(f"\nQuery: {query}")
print("Ranked results:")
scored = [(cosine_similarity(query_emb, emb), text) for emb, text in zip(embeddings, texts)]
scored.sort(reverse=True)
for score, text in scored:
    print(f"  {score:.4f} | {text}")

### Demo 2: Vector Stores — Indexing and Similarity Search with ChromaDB

A vector store indexes embeddings for fast similarity search. ChromaDB is a lightweight, in-memory vector store perfect for prototyping.

In [ ]:
# Demo 2 - Vector Stores with ChromaDB

# Step 1: Create a ChromaDB client and collection
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(
    name="demo_docs",
    metadata={"hnsw:space": "cosine"}
)

# Step 2: Add documents with metadata
documents = [
    "LangChain is a framework for building LLM applications with composable components.",
    "LangGraph extends LangChain with graph-based stateful workflows and cycles.",
    "RAG combines retrieval with generation to ground LLM answers in external knowledge.",
    "Vector stores like ChromaDB and FAISS enable fast similarity search over embeddings.",
    "Prompt engineering techniques include few-shot, chain-of-thought, and ReAct patterns.",
    "OpenAI's function calling allows LLMs to invoke external tools with structured arguments.",
    "Pydantic provides data validation and settings management using Python type annotations.",
    "Multi-agent systems use supervisor-worker patterns for complex task decomposition."
]

# Use OpenAI embeddings via the API
client = openai.OpenAI()
emb_response = client.embeddings.create(model="text-embedding-3-small", input=documents)
doc_embeddings = [item.embedding for item in emb_response.data]

collection.add(
    documents=documents,
    embeddings=doc_embeddings,
    ids=[f"doc_{i}" for i in range(len(documents))],
    metadatas=[{"source": f"chapter_{i+1}"} for i in range(len(documents))]
)
print(f"Indexed {collection.count()} documents")

# Step 3: Query the collection
query = "How do I build workflows with LLMs?"
query_emb = client.embeddings.create(model="text-embedding-3-small", input=[query]).data[0].embedding

results = collection.query(
    query_embeddings=[query_emb],
    n_results=3
)

print(f"\nQuery: {query}")
print("Top 3 results:")
for i, (doc, distance, meta) in enumerate(zip(results['documents'][0], results['distances'][0], results['metadatas'][0])):
    print(f"  {i+1}. [{meta['source']}] (dist={distance:.4f}) {doc}")

### Demo 3: Advanced Chunking Strategies

How you split documents directly affects retrieval quality. Different strategies work better for different content types: recursive splitting for general text, sentence-based for conversational content, and semantic chunking for technical documents.

In [ ]:
# Demo 3 - Advanced Chunking Strategies

sample_doc = """# Introduction to Retrieval-Augmented Generation

Retrieval-Augmented Generation (RAG) is a technique that enhances LLM responses by grounding them in external knowledge. The approach was introduced by Lewis et al. in 2020.

## How RAG Works

The RAG pipeline consists of three main stages:

1. Indexing: Documents are split into chunks, embedded, and stored in a vector database.
2. Retrieval: When a user asks a question, the query is embedded and similar chunks are found.
3. Generation: Retrieved chunks are included in the prompt, and the LLM generates an answer.

## Benefits of RAG

RAG reduces hallucinations by providing factual context. It keeps information up-to-date without retraining. It allows source attribution, improving trust and verifiability.

## Challenges

The main challenges include: chunking strategy (too small loses context, too large dilutes relevance), embedding quality, retrieval accuracy, and handling multi-hop questions that require information from multiple chunks."""

# Strategy 1: Fixed-size recursive splitting
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200, chunk_overlap=30, separators=["\n\n", "\n", ". ", " "]
)
recursive_chunks = recursive_splitter.split_text(sample_doc)

print(f"Recursive splitting: {len(recursive_chunks)} chunks")
for i, chunk in enumerate(recursive_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

# Strategy 2: Markdown-aware splitting
md_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=30,
    separators=["\n## ", "\n# ", "\n\n", "\n", ". ", " "]
)
md_chunks = md_splitter.split_text(sample_doc)

print(f"\nMarkdown-aware splitting: {len(md_chunks)} chunks")
for i, chunk in enumerate(md_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

# Strategy 3: Sentence-level splitting
sentence_splitter = RecursiveCharacterTextSplitter(
    chunk_size=150, chunk_overlap=0,
    separators=[". ", "\n", " "]
)
sentence_chunks = sentence_splitter.split_text(sample_doc)

print(f"\nSentence-level splitting: {len(sentence_chunks)} chunks")
for i, chunk in enumerate(sentence_chunks[:3]):
    print(f"  Chunk {i+1} ({len(chunk)} chars): {chunk[:80]}...")

print(f"\nComparison: Recursive={len(recursive_chunks)}, Markdown={len(md_chunks)}, Sentence={len(sentence_chunks)}")

### Demo 4: Query Transformation Techniques

Users often write queries that are too vague or specific for direct retrieval. Query transformation improves retrieval by rewriting, expanding, or hypothetically answering the query before searching.

In [ ]:
# Demo 4 - Query Transformation

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Technique 1: Multi-query expansion
def multi_query_expand(question, n=3):
    """Generate multiple alternative queries for better retrieval coverage."""
    response = llm.invoke([
        SystemMessage(content=f"Generate {n} alternative versions of this question. Each should approach it from a different angle. Return as a JSON list of strings."),
        HumanMessage(content=question)
    ])
    try:
        queries = json.loads(response.content)
    except:
        queries = [question]
    return [question] + queries

# Technique 2: HyDE (Hypothetical Document Embeddings)
def hyde_transform(question):
    """Generate a hypothetical answer, then use it for retrieval."""
    response = llm.invoke([
        SystemMessage(content="Write a short paragraph that would be a perfect answer to this question. Do not say 'I think' — write it as if it's from a textbook."),
        HumanMessage(content=question)
    ])
    return response.content

# Test both techniques
question = "How do I make my RAG system more accurate?"

print(f"Original question: {question}")
print("\n--- Multi-Query Expansion ---")
expanded = multi_query_expand(question)
for i, q in enumerate(expanded):
    print(f"  {i+1}. {q}")

print("\n--- HyDE Transform ---")
hypothetical = hyde_transform(question)
print(f"  Hypothetical answer: {hypothetical[:200]}...")
print("\n  (This hypothetical answer would be embedded and used for retrieval")
print("   instead of the original question, often finding more relevant chunks.)")

### Demo 5: End-to-End RAG Pipeline with Source Citations

This demo brings everything together into a complete RAG pipeline that retrieves relevant chunks, generates an answer, and includes source citations.

In [ ]:
# Demo 5 - End-to-End RAG Pipeline with Citations

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
embed_client = openai.OpenAI()

# Step 1: Build knowledge base
kb_docs = [
    {"text": "RAG was introduced by Lewis et al. in 2020. It combines retrieval with generation to reduce hallucinations.", "source": "rag_paper.pdf", "page": 1},
    {"text": "The indexing phase involves chunking documents, creating embeddings, and storing them in a vector database.", "source": "rag_guide.md", "page": 3},
    {"text": "ChromaDB is an open-source embedding database. It supports cosine similarity and Euclidean distance.", "source": "vectordb_docs.md", "page": 1},
    {"text": "Query transformation techniques like HyDE and multi-query improve retrieval by reformulating the question.", "source": "rag_guide.md", "page": 7},
    {"text": "Chunking strategy is critical: too small loses context, too large dilutes relevance. 200-500 tokens is typical.", "source": "best_practices.md", "page": 2},
    {"text": "Reranking uses a cross-encoder to rescore retrieved results, improving precision at the cost of latency.", "source": "rag_guide.md", "page": 9}
]

# Step 2: Index in ChromaDB
chroma = chromadb.Client()
coll = chroma.create_collection(name="rag_demo_e2e", metadata={"hnsw:space": "cosine"})

texts = [d["text"] for d in kb_docs]
embs = [e.embedding for e in embed_client.embeddings.create(model="text-embedding-3-small", input=texts).data]
coll.add(
    documents=texts, embeddings=embs,
    ids=[f"doc_{i}" for i in range(len(kb_docs))],
    metadatas=[{"source": d["source"], "page": d["page"]} for d in kb_docs]
)

# Step 3: RAG function
def rag_query(question, k=3):
    # Retrieve
    q_emb = embed_client.embeddings.create(model="text-embedding-3-small", input=[question]).data[0].embedding
    results = coll.query(query_embeddings=[q_emb], n_results=k)
    
    # Build context with source tags
    context_parts = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        context_parts.append(f"[Source: {meta['source']}, p.{meta['page']}] {doc}")
    context = "\n\n".join(context_parts)
    
    # Generate
    response = llm.invoke([
        SystemMessage(content="Answer based on the context. Cite sources in [brackets]. If unsure, say so."),
        HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}")
    ])
    
    return {"answer": response.content, "sources": results['metadatas'][0], "context": context}

# Step 4: Test
for q in ["What is RAG and who introduced it?", "How should I chunk my documents?", "What is reranking?"]:
    result = rag_query(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"Sources: {[s['source'] for s in result['sources']]}\n")

---
## Tasks (Solved)

### Task 1: Build an Embedding-Based Document Search Engine

Create a search engine that takes a corpus of documents, embeds them, and returns the most relevant documents for a query — ranked by cosine similarity.

> **Approach:** We encapsulate embedding creation and search in a class. The `__init__` method embeds all documents once, and `search` embeds the query and computes cosine similarity against all stored embeddings, returning the top-k results sorted by score.

In [ ]:
# Task 1 — SOLUTION: Embedding-Based Document Search Engine

class SearchEngine:
    def __init__(self, documents):
        self.documents = documents
        self.client = openai.OpenAI()
        # Embed all documents at init time
        response = self.client.embeddings.create(
            model="text-embedding-3-small",
            input=documents
        )
        self.embeddings = [item.embedding for item in response.data]
        print(f"Indexed {len(self.documents)} documents ({len(self.embeddings[0])} dimensions)")

    def _cosine_similarity(self, a, b):
        a, b = np.array(a), np.array(b)
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

    def search(self, query, k=3):
        # Embed the query
        query_emb = self.client.embeddings.create(
            model="text-embedding-3-small", input=[query]
        ).data[0].embedding
        
        # Compute similarity against all documents
        scored = [
            (self._cosine_similarity(query_emb, doc_emb), doc)
            for doc_emb, doc in zip(self.embeddings, self.documents)
        ]
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:k]


# Test
corpus = [
    "Python is great for machine learning and data science.",
    "JavaScript is the language of the web and runs in browsers.",
    "Rust provides memory safety without garbage collection.",
    "SQL is used to query relational databases.",
    "Docker containers package applications for consistent deployment.",
    "Kubernetes orchestrates container workloads at scale.",
    "TensorFlow and PyTorch are popular deep learning frameworks.",
    "Git is a distributed version control system."
]

engine = SearchEngine(corpus)

for query in ["What language for machine learning?", "How do I deploy applications?", "database tools"]:
    print(f"\nQuery: {query}")
    results = engine.search(query, k=3)
    for score, doc in results:
        print(f"  {score:.4f} | {doc}")

### Task 2: Implement a Multi-Strategy Chunking Pipeline

Build a chunking pipeline that applies different strategies based on document type (markdown, code, plain text) and evaluates chunk quality.

> **Approach:** We detect the document type by examining content patterns (markdown headers, Python keywords, etc.), then apply an appropriate `RecursiveCharacterTextSplitter` with type-specific separators. Each chunk is returned with metadata about its type and character count.

In [ ]:
# Task 2 — SOLUTION: Multi-Strategy Chunking Pipeline

class SmartChunker:
    def __init__(self, chunk_size=300, chunk_overlap=50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap

    def _detect_type(self, text):
        """Detect document type based on content patterns."""
        lines = text.strip().split("\n")
        header_count = sum(1 for l in lines if l.startswith("#"))
        code_keywords = ["def ", "class ", "import ", "return ", "if __name__"]
        code_count = sum(1 for kw in code_keywords if kw in text)
        
        if header_count >= 2:
            return "markdown"
        elif code_count >= 2:
            return "code"
        else:
            return "plain"

    def _get_splitter(self, doc_type):
        """Return appropriate splitter for the document type."""
        if doc_type == "markdown":
            return RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap,
                separators=["\n## ", "\n# ", "\n### ", "\n\n", "\n", ". ", " "]
            )
        elif doc_type == "code":
            return RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap,
                separators=["\nclass ", "\ndef ", "\n\n", "\n", " "]
            )
        else:
            return RecursiveCharacterTextSplitter(
                chunk_size=self.chunk_size, chunk_overlap=self.chunk_overlap,
                separators=["\n\n", "\n", ". ", " "]
            )

    def chunk(self, text):
        """Chunk text using the appropriate strategy."""
        doc_type = self._detect_type(text)
        splitter = self._get_splitter(doc_type)
        raw_chunks = splitter.split_text(text)
        
        return [
            {"text": chunk, "type": doc_type, "chars": len(chunk), "index": i}
            for i, chunk in enumerate(raw_chunks)
        ]


# Test with different document types
chunker = SmartChunker(chunk_size=200, chunk_overlap=30)

markdown_doc = """# Guide to RAG

RAG enhances LLMs with retrieval.

## Architecture

The pipeline has three stages: indexing, retrieval, and generation.

## Best Practices

Use small chunks (200-500 tokens). Overlap chunks by 10-20%. Add metadata."""

code_doc = """import openai
from typing import List

def embed_texts(texts: List[str]) -> List[List[float]]:
    client = openai.OpenAI()
    response = client.embeddings.create(model="text-embedding-3-small", input=texts)
    return [item.embedding for item in response.data]

class SearchEngine:
    def __init__(self, documents):
        self.docs = documents
        self.embeddings = embed_texts(documents)
    
    def search(self, query, k=3):
        return sorted(results, reverse=True)[:k]"""

plain_doc = """Retrieval augmented generation is a technique for improving LLM outputs. It works by finding relevant documents and including them in the prompt. This helps reduce hallucinations and keeps answers grounded in facts. Many production systems now use RAG as their primary architecture for knowledge-intensive tasks."""

for label, doc in [("Markdown", markdown_doc), ("Code", code_doc), ("Plain", plain_doc)]:
    chunks = chunker.chunk(doc)
    print(f"\n{label} ({chunks[0]['type']}): {len(chunks)} chunks")
    for c in chunks:
        print(f"  [{c['index']}] ({c['chars']} chars): {c['text'][:70]}...")

### Task 3: Create a Query Expansion and Reranking System

Build a retrieval system that expands the query into multiple variants, retrieves candidates for each, deduplicates, and reranks using the LLM.

> **Approach:** We use the multi-query expansion from Demo 4 to generate alternative formulations, retrieve from ChromaDB for each, deduplicate by document text, then use the LLM as a judge to score relevance (0-10) for each candidate. Finally, we sort by score and return the top-k.

In [ ]:
# Task 3 — SOLUTION: Query Expansion and Reranking System

class AdvancedRetriever:
    def __init__(self, collection):
        self.collection = collection
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.embed_client = openai.OpenAI()

    def _expand_query(self, question, n=3):
        """Generate alternative query formulations."""
        response = self.llm.invoke([
            SystemMessage(content=f"Generate {n} alternative versions of this question. Each should approach it from a different angle. Return as a JSON list of strings."),
            HumanMessage(content=question)
        ])
        try:
            queries = json.loads(response.content)
        except:
            queries = [question]
        return [question] + queries

    def _retrieve_candidates(self, queries, per_query_k=3):
        """Retrieve and deduplicate candidates from all query variants."""
        seen = set()
        candidates = []
        for query in queries:
            q_emb = self.embed_client.embeddings.create(
                model="text-embedding-3-small", input=[query]
            ).data[0].embedding
            results = self.collection.query(query_embeddings=[q_emb], n_results=per_query_k)
            for doc in results['documents'][0]:
                if doc not in seen:
                    seen.add(doc)
                    candidates.append(doc)
        return candidates

    def _rerank(self, question, candidates):
        """Use LLM to score relevance of each candidate."""
        scored = []
        for doc in candidates:
            response = self.llm.invoke([
                SystemMessage(content="Rate the relevance of this document to the question on a scale of 0-10. Return ONLY the number."),
                HumanMessage(content=f"Question: {question}\n\nDocument: {doc}")
            ])
            try:
                score = int(response.content.strip())
            except:
                score = 0
            scored.append((doc, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored

    def retrieve(self, query, k=3):
        """Full pipeline: expand -> retrieve -> deduplicate -> rerank."""
        print(f"Query: {query}")
        
        # Step 1: Expand
        queries = self._expand_query(query)
        print(f"Expanded to {len(queries)} queries")
        
        # Step 2: Retrieve + deduplicate
        candidates = self._retrieve_candidates(queries)
        print(f"Retrieved {len(candidates)} unique candidates")
        
        # Step 3: Rerank
        ranked = self._rerank(query, candidates)
        return ranked[:k]


# Test — reuse the collection from Demo 2
retriever = AdvancedRetriever(collection)
results = retriever.retrieve("How to improve RAG accuracy?")

print("\nReranked results:")
for doc, score in results:
    print(f"  {score}/10 | {doc[:80]}")

### Task 4: Build a Production RAG Pipeline with Evaluation Metrics

Build a complete RAG system that includes automated evaluation — measuring retrieval relevance, answer faithfulness (does the answer match the sources?), and completeness.

> **Approach:** We build a full RAG pipeline class that wraps indexing, retrieval, and generation. After generating an answer, we use LLM-as-judge to evaluate three metrics on a 1-5 scale: retrieval relevance (are chunks related to the question?), faithfulness (is the answer supported by the context?), and completeness (does the answer fully address the question?). This pattern is standard for RAG evaluation in production.

In [ ]:
# Task 4 — SOLUTION: Production RAG Pipeline with Evaluation

class EvaluatedRAG:
    def __init__(self, documents):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.embed_client = openai.OpenAI()
        
        # Index documents in ChromaDB
        self.chroma = chromadb.Client()
        self.coll = self.chroma.create_collection(
            name="evaluated_rag", metadata={"hnsw:space": "cosine"}
        )
        embs = [e.embedding for e in self.embed_client.embeddings.create(
            model="text-embedding-3-small", input=documents
        ).data]
        self.coll.add(
            documents=documents, embeddings=embs,
            ids=[f"doc_{i}" for i in range(len(documents))]
        )
        print(f"Indexed {len(documents)} documents")

    def query(self, question, k=3):
        """Retrieve, generate, and evaluate."""
        # Retrieve
        q_emb = self.embed_client.embeddings.create(
            model="text-embedding-3-small", input=[question]
        ).data[0].embedding
        results = self.coll.query(query_embeddings=[q_emb], n_results=k)
        context = "\n\n".join(results['documents'][0])
        
        # Generate
        response = self.llm.invoke([
            SystemMessage(content="Answer based on the context. If unsure, say so."),
            HumanMessage(content=f"Context:\n{context}\n\nQuestion: {question}")
        ])
        answer = response.content
        
        # Evaluate
        metrics = self.evaluate(question, answer, context)
        
        return {"answer": answer, "context": context, "metrics": metrics}

    def evaluate(self, question, answer, context):
        """Evaluate RAG quality using LLM-as-judge."""
        metrics = {}
        
        eval_prompts = {
            "relevance": f"Rate 1-5: How relevant is this context to the question?\n\nQuestion: {question}\n\nContext: {context}",
            "faithfulness": f"Rate 1-5: Is the answer fully supported by the context (no hallucinations)?\n\nContext: {context}\n\nAnswer: {answer}",
            "completeness": f"Rate 1-5: Does the answer fully address the question?\n\nQuestion: {question}\n\nAnswer: {answer}"
        }
        
        for metric_name, prompt in eval_prompts.items():
            response = self.llm.invoke([
                SystemMessage(content="You are a strict evaluator. Return ONLY a number 1-5."),
                HumanMessage(content=prompt)
            ])
            try:
                metrics[metric_name] = int(response.content.strip())
            except:
                metrics[metric_name] = 3  # default
        
        metrics["overall"] = round(sum(metrics.values()) / len(metrics), 2)
        return metrics


# Test
docs = [
    "RAG was introduced by Lewis et al. in 2020 to reduce LLM hallucinations.",
    "Chunking strategy is critical: 200-500 tokens per chunk is recommended.",
    "HyDE generates hypothetical answers to improve retrieval quality.",
    "Reranking rescores retrieved documents using a cross-encoder model.",
    "ChromaDB and FAISS are popular vector stores for RAG systems."
]

rag = EvaluatedRAG(docs)

for q in ["What is RAG?", "How should I chunk documents?", "What is HyDE?"]:
    result = rag.query(q)
    print(f"Q: {q}")
    print(f"A: {result['answer'][:120]}...")
    print(f"Metrics: {result['metrics']}\n")

---
## Summary

In this session students learned production-grade RAG engineering:

1. **Embeddings** -- How text is converted to vectors where semantic similarity equals spatial proximity.
2. **Vector Stores** -- How ChromaDB indexes embeddings for sub-millisecond similarity search.
3. **Chunking Strategies** -- How splitting strategy directly affects retrieval quality.
4. **Query Transformation** -- How multi-query and HyDE improve retrieval for complex questions.
5. **End-to-End RAG** -- How to combine retrieval, generation, and source citations into a production pipeline.

**Up Next:** In Session 2, we will learn how to deploy and scale these systems in production.